In [ ]:
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

In [ ]:
!scikit-learn

### Prepare mask

In [ ]:
oil_mask = gpd.read_file('oil_mask_31052020.geojson')
oil_mask.plot()

In [ ]:
oil_mask

In [ ]:
src = rasterio.open('data/Norilsk_oil_spill/31-05-2020.tiff')
bands = src.read()

In [ ]:
from rasterio.mask import mask


oil_bands, oil_bands_transform = mask(src, oil_mask[['geometry']].values.flatten())

In [ ]:
oil_bands.shape

In [ ]:
plt.imshow(oil_bands[0, :, :])
plt.colorbar()

In [ ]:
oil_bands

In [ ]:
non_oil_bands = np.where(oil_bands == 0, bands, 0)

non_oil_bands = np.where(non_oil_bands == 0, -1, non_oil_bands)
oil_bands = np.where(oil_bands == 0, -1, oil_bands)

In [ ]:
plt.imshow(non_oil_bands[3, :, :])
plt.colorbar()

In [ ]:
# np.zeros(non_oil_bands.shape) + 1

class_mask = np.where(non_oil_bands != -1, 1, -1)
oil_mask = np.where(oil_bands != -1, 2, -1)
class_mask = class_mask + oil_mask

class_mask = class_mask[0, :, :]
class_mask = np.expand_dims(class_mask, axis=0)
class_mask = np.where(class_mask < 0, np.nan, class_mask)

data = np.concatenate(
    (bands, class_mask),
    axis=0)

In [ ]:
data.shape

In [ ]:
np.unique(class_mask)

In [ ]:
plt.imshow(data[14, :, :])
plt.colorbar()

In [ ]:
df = data.reshape([15, data.shape[1] * data.shape[2]])
df = pd.DataFrame(
    df.T,
    columns=["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09", "B11","B12","SCL","CLM", "isOil"])
df.dropna(inplace=True)

In [ ]:
df

### Prepare dataset

In [ ]:
features = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09',
       'B11', 'B12', 'SCL', 'CLM']

scaler = StandardScaler()
data = scaler.fit_transform(df[features].copy())

In [ ]:
from imblearn.over_sampling import SMOTE

ov_train_x, ov_train_y = SMOTE().fit_resample(data, df.isOil)
ov_train_y = ov_train_y.astype('int')

ov_train_y.value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(ov_train_x, ov_train_y, test_size=0.33, random_state=42)

### Models

In [ ]:
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix, roc_curve, auc, roc_curve,accuracy_score
import seaborn as sns


def report(y_test, y_pred):
    print('Validation accuracy is', accuracy_score(y_pred,y_test))
    print ("\nClassification report :\n",(classification_report(y_test,y_pred)))

    #Confusion matrix
    plt.figure(figsize=(13,10))
    plt.subplot(221)
    sns.heatmap(confusion_matrix(y_test,y_pred),annot=True,cmap="viridis",fmt = "d",linecolor="k",linewidths=3)
    plt.title("CONFUSION MATRIX",fontsize=20)

    #ROC curve and Area under the curve plotting
    predicting_probabilites = y_pred
    fpr,tpr,thresholds = roc_curve(y_test,predicting_probabilites)
    plt.subplot(222)
    plt.plot(fpr,tpr,label = ("Area_under the curve :",auc(fpr,tpr)),color = "r")
    plt.plot([1,0],[1,0],linestyle = "dashed",color ="k")
    plt.legend(loc = "best")
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title("ROC - CURVE & AREA UNDER CURVE",fontsize=20)

In [ ]:
from sklearn.svm import SVC

In [ ]:
# svm_model = SVC(gamma='auto')

# svm_model.fit(X_train, y_train)

In [ ]:
# y_pred = svm_model.predict(X_test)

In [ ]:
# report(y_test, y_pred)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

In [ ]:
ada_model = AdaBoostClassifier(n_estimators=100, random_state=0)

ada_model.fit(X_train, y_train)

In [ ]:
y_pred = ada_model.predict(X_test)

In [ ]:
report(y_test, y_pred)

In [ ]:
from sklearn.tree import DecisionTreeClassifier


dt_model = DecisionTreeClassifier(max_depth=5, random_state=13)
dt_model.fit(X_train, y_train)
y_pred=dt_model.predict(X_test)

In [ ]:
report(y_test, y_pred)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(max_depth=2, random_state=0)

rf_model.fit(X_train, y_train)

In [ ]:
y_pred = rf_model.predict(X_test)
y_pred

In [ ]:
report(y_test, y_pred)